# Healthcare Fraud Detection

## Business Question

> _What patterns most reliably identify fraudulent insurance claims?_

## Dataset

> Sourced from [Kaggle](https://www.kaggle.com/datasets/nudratabbas/healthcare-fraud-detection-dataset/data), this dataset comprises 10,000 simulated healthcare insurance claims that model real-world fraud scenarios. It includes a range of patient, provider, and financial attributes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## Load Data

In [ ]:
fraud_df = pd.read_csv("healthcare_fraud_detection.csv")

## Raw Data Overview

In [ ]:
df = fraud_df
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
df.info()
print(f"\nMissing values per column:\n{df.isnull().sum()}")
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")

print(f"\nFraud breakdown:\n{df['Is_Fraud'].value_counts()}")
print(f"\nFraud rate: {df['Is_Fraud'].mean():.2%}")

## Clean Data

In [ ]:
df["Claim_Submission_Date"] = pd.to_datetime(df["Claim_Submission_Date"])

In [ ]:
df["Insurance_Type"] = df["Insurance_Type"].fillna("Unknown")
df["Provider_Specialty"] = df["Provider_Specialty"].fillna("Unknown")

In [ ]:
print(df["Prior_Visits_12m"].describe())
print(f"Skewness: {df['Prior_Visits_12m'].skew()}")

In [ ]:
df["Prior_Visits_12m"] = df["Prior_Visits_12m"].fillna(df["Prior_Visits_12m"].median())

In [ ]:
df.info()

## Feature Engineering

In [ ]:
print(df["Number_of_Claims_Per_Provider_Monthly"].describe())

In [ ]:
# Flag providers with more than 90 monthly claims
df["High_Volume_Claims_Flag"] = (df["Number_of_Claims_Per_Provider_Monthly"] > 90).astype(int)

In [ ]:
# Extract components from date column
df["Month"] = df["Claim_Submission_Date"].dt.month_name()
df["Quarter"] = "Q" + df["Claim_Submission_Date"].dt.quarter.astype(str)
df["Year"] = df["Claim_Submission_Date"].dt.year
df

## Analysis

In [ ]:
fraud_rate = df["Is_Fraud"].mean()

In [ ]:
# Function to return fraud rate, fraud claims and claim count for a column.
def fraud_by (column):
    return (
        df.groupby(column)["Is_Fraud"]
            .agg(
                Fraud_Rate = "mean"
                ,Fraud_Claims = "sum"
                ,Total_Claims = "count"
        )
        .sort_values("Fraud_Rate", ascending=False)
        .reset_index()
    )

In [ ]:
# Fraud by provider specialty
fraud_by_provider_specialty = fraud_by("Provider_Specialty")

fraud_by_provider_specialty

In [ ]:
# Fraud by insurance type
fraud_by_insurance_type = fraud_by("Insurance_Type")

fraud_by_insurance_type

In [ ]:
# Fraud by visit type
fraud_by_visit_type = fraud_by("Visit_Type")

fraud_by_visit_type 

## Visualisations

In [ ]:
# Fraud rate by provider specialty
plt.figure()

plt.barh(
    fraud_by_provider_specialty["Provider_Specialty"]
    ,fraud_by_provider_specialty["Fraud_Rate"] * 100
)

plt.axvline(
    x = fraud_rate * 100
    ,linestyle="--"
    ,label=f"Overall Avg Fraud Rate ({fraud_rate:.1%})"
)

plt.legend()

plt.title("Fraud Rate by Provider Specialty")

plt.xlabel("Fraud Rate (%)")
plt.ylabel("Provider Specialty")

plt.tight_layout()

plt.savefig("charts/fraud_rate_by_specialty.png")

In [ ]:
# Fraud rate by insurance type
plt.figure()

plt.barh(
    fraud_by_provider_specialty["Insurance_Type"]
    ,fraud_by_provider_specialty["Fraud_Rate"] * 100
)

plt.axvline(
    x = fraud_rate * 100
    ,linestyle="--"
    ,label=f"Overall Avg Fraud Rate ({fraud_rate:.1%})"
)

plt.legend()

plt.title("Fraud Rate by Insurance Type")

plt.xlabel("Fraud Rate (%)")
plt.ylabel("Insurance_Type")

plt.tight_layout()

plt.savefig("charts/fraud_rate_by_insurance_type.png")